In [ ]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [2]:
DATA_PATH = Path("../news_data/cleaned_news_for_model.parquet")

df = pd.read_parquet(DATA_PATH)

df.shape, df.head()

((146619, 10),
      source                                                url archive_date  \
 0  lenta.ru             https://lenta.ru/news/2025/01/01/auto/   2025-01-01   
 1  lenta.ru  https://lenta.ru/news/2025/01/01/v-rossii-stal...   2025-01-01   
 2  lenta.ru  https://lenta.ru/news/2025/01/01/v-rossii-s-1-...   2025-01-01   
 3  lenta.ru  https://lenta.ru/news/2025/01/01/vsu-v-pervye-...   2025-01-01   
 4  lenta.ru  https://lenta.ru/news/2025/01/01/premier-mishu...   2025-01-01   
 
          published_at                                              title  \
 0 2025-01-01 00:00:00  Рост акцизов на топливо, увеличение утильсбора...   
 1 2025-01-01 00:02:00                   В России стало дороже развестись   
 2 2025-01-01 01:23:00  В России с 1 января повысили штрафы за нарушен...   
 3 2025-01-01 01:43:00  ВСУ в первые минуты нового года обстреляли рос...   
 4 2025-01-01 01:57:00   Премьер Мишустин поздравил россиян с Новым годом   
 
                                       

In [3]:
df_model = df[["title", "text", "category_raw"]].dropna().copy()

df_model["bert_text"] =  df_model["text"].astype(str) #.str[:1500]


df_model = df_model[df_model["bert_text"].str.len() > 0]

df_model.shape

(146619, 4)

In [4]:
RUN_ON_SAMPLE = False
SAMPLE_SIZE = 20000

if RUN_ON_SAMPLE:
    df_exp, _ = train_test_split(
        df_model,
        train_size=SAMPLE_SIZE,
        random_state=42,
        stratify=df_model["category_raw"]
    )
else:
    df_exp = df_model.copy()

df_exp["category_raw"].value_counts()

category_raw
Мир                45383
Россия             41352
Экономика          32112
Наука и техника    10168
Спорт               9869
Культура            7735
Name: count, dtype: int64

In [5]:
X = df_exp["bert_text"]
y = df_exp["category_raw"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((117295,), (29324,))

In [6]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedder = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 4529.49it/s]


In [7]:
X_train_emb = embedder.encode(
    X_train.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_test_emb = embedder.encode(
    X_test.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_train_emb.shape, X_test_emb.shape

Batches: 100%|███████████████████████████████████████████████████████████████████████| 459/459 [19:59<00:00,  2.61s/it]


((117295, 384), (29324, 384))

In [8]:
import os

os.makedirs("../news_data/interim", exist_ok=True)

np.save("../news_data/interim/Exp_02_X_train_sbert_minilm.npy", X_train_emb)
np.save("../news_data/interim/Exp_02_X_test_sbert_minilm.npy", X_test_emb)

y_train.to_csv("../news_data/interim/Exp_02_y_train_sbert_minilm.csv", index=False)
y_test.to_csv("../news_data/interim/Exp_02_y_test_sbert_minilm.csv", index=False)

In [9]:
clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)

In [10]:
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.8680261901514118
Macro F1: 0.8742976426584009
Weighted F1: 0.8678625069618524


In [11]:
print(classification_report(y_test, y_pred))

                 precision    recall  f1-score   support

       Культура       0.78      0.95      0.85      1547
            Мир       0.90      0.88      0.89      9077
Наука и техника       0.77      0.93      0.85      2034
         Россия       0.87      0.80      0.84      8270
          Спорт       0.94      0.97      0.96      1974
      Экономика       0.86      0.86      0.86      6422

       accuracy                           0.87     29324
      macro avg       0.85      0.90      0.87     29324
   weighted avg       0.87      0.87      0.87     29324



In [12]:
result = pd.DataFrame([{
    "experiment": "02_sbert_minilm_text_logreg",
    "model": MODEL_NAME,
    "input": "first_1500_chars_text",
    "classifier": "LogisticRegression",
    "sample_size": len(df_exp),
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1
}])

os.makedirs("../reports", exist_ok=True)

result.to_csv("../reports/sbert_minilm_results.csv", index=False)

result

,experiment,model,input,classifier,sample_size,accuracy,macro_f1,weighted_f1
0,02_sbert_minilm_title_text_logreg,sentence-transformers/paraphrase-multilingual-...,first_1500_chars_text,LogisticRegression,146619,0.868026,0.874298,0.867863


In [13]:
labels = sorted(y_test.unique())

cm = confusion_matrix(y_test, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

cm_df

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,1462,14,4,39,7,21
true_Мир,86,7986,195,516,24,270
true_Наука и техника,6,49,1899,45,4,31
true_Россия,199,638,185,6643,56,549
true_Спорт,12,8,4,15,1924,11
true_Экономика,115,220,164,360,23,5540


In [14]:
cm_norm = cm / cm.sum(axis=1, keepdims=True)

cm_norm_df = pd.DataFrame(
    cm_norm,
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

cm_norm_df.round(3)

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,0.945,0.009,0.003,0.025,0.005,0.014
true_Мир,0.009,0.880,0.021,0.057,0.003,0.030
true_Наука и техника,0.003,0.024,0.934,0.022,0.002,0.015
true_Россия,0.024,0.077,0.022,0.803,0.007,0.066
true_Спорт,0.006,0.004,0.002,0.008,0.975,0.006
true_Экономика,0.018,0.034,0.026,0.056,0.004,0.863


In [15]:
errors = pd.DataFrame({
    "text": X_test,
    "true": y_test,
    "pred": y_pred
})

errors = errors[errors["true"] != errors["pred"]]

errors.head(20)

,text,true,pred
93958,Юрист Кваша: На шумящих по ночам соседей можно...,Экономика,Россия
42845,Правительство одобрило увеличение пособия по б...,Россия,Экономика
33921,Политолог Виноградов пошутил о сакральности по...,Россия,Мир
140610,Профайлер Панкратов заметил странности в повед...,Мир,Россия
79379,В Сириусе Путину показали клубнику и виноград ...,Наука и техника,Экономика
131084,"Минтранс сообщил, что ВСУ атаковали БЭКами рос...",Мир,Россия
79924,ДОМ.РФ профинансировал реконструкцию троллейбу...,Россия,Экономика
41823,Мединский: Россия будет ждать украинскую делег...,Россия,Мир
97662,Эксперт Киселев: Зеленского в Польше может жда...,Мир,Спорт
66558,ВТБ открыл силовым ведомствам доступ к програм...,Экономика,Россия
